#### Real-time AI assistant that can answer questions from documents, websites, PDFs, notes, or knowledge bases


##### Uses DuckDuckGo search with LLM

In [1]:
import os
from huggingface_hub import InferenceClient
from ddgs import DDGS

/root/opt/la-i-b/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setup the InferenceClient with the Hugging Face token

client = InferenceClient(token=os.getenv("HF_TOKEN"))
MODEL = "Qwen/Qwen2.5-7B-Instruct"

def ask_llm(prompt, max_tokens=500):
    result = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL,
        max_tokens=max_tokens
    )
    return result.choices[0].message.content.strip()

In [3]:
def web_search(query, max_results=5):
    """
    Performs a web search using DuckDuckGo and returns the results.
    """
    try:
        results = DDGS().text(query, max_results=max_results)
        if not results:
            return "No search results found."
        
        # Debug - see what we're getting
        for r in results:
            print(f"  Found: {r['title']}")
        
        return "\n\n".join(
            f"Title: {r['title']}\n{r['body']}" for r in results
        )
    except Exception as e:
        return f"Search failed: {e}"

In [4]:
def rag_answer(question):
    """ Real-time RAG function that searches the web and feeds the results to the LLM """
    # RETRIEVE - search the web
    print("Searching the web...")
    context = web_search(question)

    # AUGMENT + GENERATE - feed context to LLM
    prompt = f"""You are a helpful AI assistant. Answer the user's question
                based *only* on the following search results. If the search results
                are empty or do not contain the answer, say 'I could not find
                any information on that.'

                Search Results:
                {context}

                Question:
                {question}"""

    return ask_llm(prompt)

In [5]:
print("Hello! I'm a real-time AI assistant. Ask me anything!")

while True:
    try:
        user_query = input("\nYou: ")
        if user_query.lower() in ["exit", "quit"]:
            print("Goodbye!")
            break

        print("Thinking...")
        answer = rag_answer(user_query)
        print(f"\nAssistant: {answer}")

    except Exception as e:
        print(f"Error: {e}")

Hello! I'm a real-time AI assistant. Ask me anything!
Thinking...
Searching the web...
  Found: Erling Haaland - Wikipedia
  Found: Erling Haaland | 2026 FIFA World Cup, Norway, Manchester City, Stats ...
  Found: Erling Haaland height, age, girlfriend, stats, Norway World Cup run
  Found: Erling Haaland - Profile, News & Videos | Manchester City F.C.
  Found: Why everyone loves Erling Haaland, Norway's giant blonde striker

Assistant: Erling Haaland is a Norwegian professional footballer who plays as a striker. He is currently a player for Premier League club Manchester City and the Norway national team. Known for his speed, strength, positioning, and finishing inside the box, Haaland is regarded as one of the best players in the world and the greatest Norwegian player of all time. He began his senior career with Bryne FK and has since played for Molde, Red Bull Salzburg, and Borussia Dortmund before joining Manchester City. Haaland has also been a key player for the Norwegian nationa